# 20 — Améliorations du modèle

Le notebook 19 donne **0,825** sur la cohorte élargie (1 185 participants, cible à deux ans).
Ce notebook applique quatre changements et **mesure l'apport de chacun séparément**,
pour pouvoir dire en soutenance lesquels ont servi et lesquels non.

| # | Changement | Pourquoi |
|---|---|---|
| A | **Indicateurs de présence** | 83 % des biomarqueurs manquent ; le fait d'avoir eu l'examen est en soi une information |
| B | **Gradient boosting** à valeurs manquantes natives | évite d'inventer une médiane là où 83 % des valeurs manquent |
| C | **Modèles séparés CN et MCI** | deux populations à 7 % et 25 % de risque, prédicteurs différents |
| D | **Utiliser le temps** | la question 2 a déjà montré +0,053 en passant de 1 à 3 visites |

Les sections A, B et C se comparent directement au 0,825, car elles utilisent
**la même cohorte, la même cible et les mêmes plis**.

La section D construit une **cohorte différente** (il faut au moins 3 visites), elle a
donc sa propre comparaison interne : même cohorte, une visite contre trois visites.

**Exécution :** noyau Python de `E:\Projet_Alzheimer\.venv`, après les notebooks 18 et 19.

## 1. Repartir exactement du notebook 19

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import platform
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
TABLES = ROOT / "data" / "02_Tables_originales"
MANIFESTES = ROOT / "data" / "05_Manifestes"

dossiers = sorted(MANIFESTES.glob("extension_tabulaire_18_*"))
if not dossiers:
    raise FileNotFoundError("Aucun export du notebook 18 : lance-le d'abord.")
EXPORT18 = dossiers[-1]

cohorte = pd.read_csv(EXPORT18 / "cohorte_tabulaire_elargie.csv", encoding="utf-8-sig")
cohorte["date_T0"] = pd.to_datetime(cohorte["date_T0"], errors="coerce")
CIBLE = "cible_aggravation_2ans"

# --- age : meme recuperation que dans le notebook 19 ---
demo_path = sorted(TABLES.rglob("All_Subjects_PTDEMOG_*.csv"))
age_recupere = False
if demo_path:
    demo = pd.read_csv(demo_path[-1], dtype="string", encoding="utf-8-sig")
    demo.columns = demo.columns.str.strip().str.upper()
    demo["PTID"] = demo["PTID"].str.strip().str.upper()
    col = next((c for c in ("PTDOBYY", "PTDOB", "PTDOBYEAR") if c in demo.columns), None)
    if col:
        d = demo.loc[demo["PTID"].isin(set(cohorte["PTID"])), ["PTID", col]].copy()
        d[col] = d[col].astype("string").str.strip()
        d = d.loc[d[col].notna() & d[col].ne("")].drop_duplicates("PTID", keep="first")
        d["naissance"] = pd.to_numeric(d[col].str.extract(r"(\d{4})")[0], errors="coerce")
        d = d.loc[d["naissance"].between(1850, 2026), ["PTID", "naissance"]]
        cohorte = cohorte.merge(d, on="PTID", how="left", validate="one_to_one")
        cohorte["age_reference"] = cohorte["date_T0"].dt.year - cohorte["naissance"]
        cohorte.loc[~cohorte["age_reference"].between(40, 120), "age_reference"] = np.nan
        age_recupere = bool(cohorte["age_reference"].notna().any())

FUITES = {"dx_cible", "date_cible", "delai_jours", "suivi_observe_jours",
          "reversion_a_la_visite_cible", CIBLE}
EXCLURE_EXACT = {"PTID", "statut", "nb_variables_presentes", "exploitable_tabulaire",
                 "annee_naissance", "naissance", "present_ancienne_cohorte"} | FUITES
EXCLURE_PREFIXE = ("date_", "anciennete_", "Unnamed")

variables = [c for c in cohorte.columns
             if c not in EXCLURE_EXACT and not c.startswith(EXCLURE_PREFIXE)
             and pd.api.types.is_numeric_dtype(cohorte[c]) and cohorte[c].notna().any()]

X = cohorte[variables].to_numpy(dtype=float)
y = cohorte[CIBLE].to_numpy(dtype=int)

print("Export 18 :", EXPORT18.name)
print(f"Participants : {len(y)} | positifs : {y.sum()} ({y.mean():.1%})")
print(f"Age recupere : {age_recupere} | {len(variables)} variables")
print(variables)

## 2. Le protocole de comparaison

Toutes les variantes passent par la **même fonction d'évaluation**, les **mêmes 5 plis**
et le même `random_state`. C'est la seule façon de dire qu'un écart vient du changement
testé et pas du découpage.

La ligne de référence est le meilleur modèle du notebook 19.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def p_logreg(Xa, ya, Xt):
    m = Pipeline([("imp", SimpleImputer(strategy="median")),
                  ("ech", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])
    return m.fit(Xa, ya).predict_proba(Xt)[:, 1]

def p_foret(Xa, ya, Xt):
    m = Pipeline([("imp", SimpleImputer(strategy="median")),
                  ("clf", RandomForestClassifier(n_estimators=300, random_state=0,
                                                 class_weight="balanced", n_jobs=-1))])
    return m.fit(Xa, ya).predict_proba(Xt)[:, 1]

def p_boosting(Xa, ya, Xt):
    """Gradient boosting : gere les valeurs manquantes sans imputation."""
    m = HistGradientBoostingClassifier(random_state=0, max_iter=300,
                                       learning_rate=0.06, max_leaf_nodes=15,
                                       early_stopping=False)
    m.fit(Xa, ya, sample_weight=compute_sample_weight("balanced", ya))
    return m.predict_proba(Xt)[:, 1]

MODELES = {"Regression logistique": p_logreg,
           "Random forest": p_foret,
           "Gradient boosting": p_boosting}

def evaluer(Xm, ym, noms=("Regression logistique", "Random forest"), cv=CV):
    """AUC moyenne par modele + ensemble (moyenne des probabilites), memes plis."""
    auc = {n: [] for n in noms}
    auc["Ensemble"] = []
    oof = {n: np.full(len(ym), np.nan) for n in list(noms) + ["Ensemble"]}
    for app, test in cv.split(Xm, ym):
        p = {n: MODELES[n](Xm[app], ym[app], Xm[test]) for n in noms}
        p["Ensemble"] = np.mean([p[n] for n in noms], axis=0)
        for n in p:
            auc[n].append(roc_auc_score(ym[test], p[n]))
            oof[n][test] = p[n]
    return auc, oof

def resume(auc):
    return {n: (float(np.mean(v)), float(np.std(v))) for n, v in auc.items()}

# --- reference : notebook 19 ---
auc_ref, oof_ref = evaluer(X, y)
REFERENCE = max(resume(auc_ref).items(), key=lambda kv: kv[1][0])
print("Reference (notebook 19) :")
for n, (m, s) in resume(auc_ref).items():
    print(f"  {n:24s} {m:.3f} +/- {s:.3f}")
print(f"\n-> meilleur : {REFERENCE[0]} = {REFERENCE[1][0]:.3f}")
comparaison = [{"variante": "A0 — notebook 19 (reference)", "modele": REFERENCE[0],
                "roc_auc": round(REFERENCE[1][0], 3), "ecart_type": round(REFERENCE[1][1], 3),
                "ecart_vs_reference": 0.0, "participants": len(y)}]

## 3. Changement A — indicateurs de présence

Dans ADNI, **le fait d'avoir eu un examen n'est pas le fruit du hasard** : un patient
qui inquiète est plus souvent envoyé en ponction lombaire ou testé à l'ADAS13. Remplacer
une valeur absente par la médiane efface cette information.

On ajoute donc, pour chaque variable manquante à plus de 30 %, une colonne binaire
« cette mesure a-t-elle été faite ? ». Elle est connue à T0, elle ne vient pas du futur.

C'est gratuit en temps de calcul et c'est le genre de détail qui plaît à un jury :
ça montre qu'on a réfléchi à la *nature* des valeurs manquantes.

In [ ]:
SEUIL_RARE = 0.30
rares = [c for c in variables if cohorte[c].isna().mean() > SEUIL_RARE]
print("Variables manquantes a plus de 30 % :", rares)

cohorte_A = cohorte.copy()
indicateurs = []
for c in rares:
    nom = "presence_" + c
    cohorte_A[nom] = cohorte_A[c].notna().astype(int)
    indicateurs.append(nom)
# une seule colonne pour les 3 biomarqueurs s'ils vont toujours ensemble
variables_A = variables + indicateurs
X_A = cohorte_A[variables_A].to_numpy(dtype=float)

print(f"{len(indicateurs)} indicateurs ajoutes -> {len(variables_A)} variables")
print("\nTaux d'aggravation selon la presence de la mesure :")
for c in rares:
    t = cohorte_A.groupby(cohorte_A[c].notna())[CIBLE].agg(["mean", "size"])
    t.index = ["mesure absente", "mesure presente"]
    print(f"\n  {c} :")
    print("   ", " | ".join(f"{i} : {r['mean']:.1%} sur {int(r['size'])}"
                            for i, r in t.iterrows()))

auc_A, _ = evaluer(X_A, y)
rA = resume(auc_A)
best_A = max(rA.items(), key=lambda kv: kv[1][0])
print(f"\n=== A — avec indicateurs de presence ===")
for n, (m, s) in rA.items():
    print(f"  {n:24s} {m:.3f} +/- {s:.3f}")
print(f"-> {best_A[0]} = {best_A[1][0]:.3f}  "
      f"({best_A[1][0] - REFERENCE[1][0]:+.3f} vs reference)")
comparaison.append({"variante": "A — indicateurs de presence", "modele": best_A[0],
                    "roc_auc": round(best_A[1][0], 3), "ecart_type": round(best_A[1][1], 3),
                    "ecart_vs_reference": round(best_A[1][0] - REFERENCE[1][0], 3),
                    "participants": len(y)})

## 4. Changement B — gradient boosting à valeurs manquantes natives

`HistGradientBoostingClassifier` sait traiter une valeur absente **comme une catégorie
à part entière** : à chaque coupure de l'arbre, il apprend de quel côté envoyer les
valeurs manquantes. Aucune médiane inventée.

Sur des données aussi creuses que les tiennes (83 % de valeurs manquantes sur les
biomarqueurs), c'est souvent ce qui fait la différence avec une imputation.

On le teste sur les variables d'origine **et** sur celles avec indicateurs.

In [ ]:
for libelle, Xm, vars_ in [("B — boosting (variables d'origine)", X, variables),
                           ("B+ — boosting + indicateurs", X_A, variables_A)]:
    auc_B, oof_B = evaluer(Xm, y, noms=("Regression logistique", "Gradient boosting"))
    rB = resume(auc_B)
    best_B = max(rB.items(), key=lambda kv: kv[1][0])
    print(f"=== {libelle} ({len(vars_)} variables) ===")
    for n, (m, s) in rB.items():
        print(f"  {n:24s} {m:.3f} +/- {s:.3f}")
    print(f"-> {best_B[0]} = {best_B[1][0]:.3f}  "
          f"({best_B[1][0] - REFERENCE[1][0]:+.3f} vs reference)\n")
    comparaison.append({"variante": libelle, "modele": best_B[0],
                        "roc_auc": round(best_B[1][0], 3),
                        "ecart_type": round(best_B[1][1], 3),
                        "ecart_vs_reference": round(best_B[1][0] - REFERENCE[1][0], 3),
                        "participants": len(y)})
    if libelle.startswith("B+"):
        oof_meilleur_tabulaire = oof_B[best_B[0]]

## 5. Changement C — traiter CN et MCI séparément

Les deux groupes n'ont ni le même risque (7 % contre 25 %) ni forcément les mêmes
prédicteurs. Deux stratégies possibles :

- **un seul modèle** pour tout le monde, avec `dx_T0` comme variable — c'est ce qu'on
  fait depuis le début ;
- **un modèle par groupe**, entraîné uniquement sur les patients de ce groupe.

On compare les deux, **à l'intérieur de chaque groupe** : c'est la seule comparaison
honnête, puisque séparer les groupes enlève mécaniquement la part de la note qui venait
du simple fait de distinguer CN de MCI.

In [ ]:
lignes_C = []
for code, nom in [(1, "CN"), (2, "MCI")]:
    m = (cohorte["dx_T0"] == code).to_numpy()
    y_g = y[m]
    if m.sum() < 60 or y_g.sum() < 15:
        print(f"{nom} : trop peu d'evenements, groupe ignore.")
        continue

    # a) modele global, lu dans ce groupe
    auc_global = roc_auc_score(y_g, oof_ref[REFERENCE[0]][m])

    # b) modele dedie, validation croisee a l'interieur du groupe
    vars_g = [c for c in variables_A if c != "dx_T0"]
    X_g = cohorte_A.loc[m, vars_g].to_numpy(dtype=float)
    cv_g = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    auc_g, _ = evaluer(X_g, y_g, noms=("Regression logistique", "Gradient boosting"), cv=cv_g)
    best_g = max(resume(auc_g).items(), key=lambda kv: kv[1][0])

    lignes_C.append({"groupe": nom, "participants": int(m.sum()),
                     "positifs": int(y_g.sum()),
                     "taux_positifs": round(float(y_g.mean()), 3),
                     "modele global (lu dans le groupe)": round(auc_global, 3),
                     "modele dedie au groupe": round(best_g[1][0], 3),
                     "meilleur modele dedie": best_g[0]})

sous_groupes = pd.DataFrame(lignes_C)
display(sous_groupes)
print("Lecture : si « modele dedie » depasse « modele global », il vaut mieux "
      "entrainer un modele par groupe.")
print("Rappel : ces notes ne se comparent PAS au 0,825, qui porte sur les deux groupes "
      "melanges.")

## 6. Changement D — utiliser le temps

C'est le levier le plus prometteur, parce que tu l'as déjà démontré : dans la question 2,
passer d'une visite à trois visites plus les pentes avait apporté **+0,053**, en gagnant
sur les 5 découpages sur 5.

Le notebook 19 n'utilise qu'une seule visite. On construit donc ici une cohorte
longitudinale sur les mêmes tables :

- on prend les **3 premières visites diagnostiques** valides de chaque participant ;
- le point d'ancrage **Tk** est la date de la 3ᵉ visite ;
- la cible est le diagnostic à la visite la plus proche de **Tk + 730 jours**, dans la
  fenêtre [640, 820] — exactement la règle du notebook 18, simplement décalée ;
- on exclut ceux déjà déments à Tk ;
- les variables sont mesurées **à chaque visite**, jamais après Tk.

Deux modèles sur **cette même cohorte** :
**statique** (la 3ᵉ visite seulement) contre **longitudinal** (les 3 visites + les pentes).

> La cohorte n'est plus la même qu'en section 3-5 : la comparaison se lit **à l'intérieur
> de cette section**, pas contre le 0,825.

In [ ]:
K = 3
FENETRE = (640, 820)
HORIZON = 730
SCORES = ["MMSE", "CDR_global", "CDR_SB", "ADAS13", "FAQ"]

dx_files = sorted(TABLES.rglob("All_Subjects_DXSUM_*.csv"))
if not dx_files:
    raise FileNotFoundError("Table DXSUM introuvable.")

dxb = pd.read_csv(dx_files[-1], dtype="string", encoding="utf-8-sig")
dxb.columns = dxb.columns.str.strip().str.upper()
dxb["PTID"] = dxb["PTID"].str.strip().str.upper()
dxb["date_dx"] = pd.to_datetime(dxb["EXAMDATE"].astype("string").str.strip().str.slice(0, 10),
                                format="%Y-%m-%d", errors="coerce").dt.normalize()
dxb["code_dx"] = pd.to_numeric(dxb["DIAGNOSIS"], errors="coerce")
dxb = dxb.loc[dxb["PTID"].str.fullmatch(r"\d{3}_S_\d{4,}", na=False)
              & dxb["date_dx"].notna() & dxb["code_dx"].isin([1, 2, 3]),
              ["PTID", "date_dx", "code_dx"]]
# memes regles que le notebook 18 : visites contradictoires ecartees
n_codes = dxb.groupby(["PTID", "date_dx"])["code_dx"].nunique()
conflits = n_codes[n_codes > 1].reset_index()[["PTID", "date_dx"]]
dxb = dxb.loc[~pd.MultiIndex.from_frame(dxb[["PTID", "date_dx"]]).isin(
    pd.MultiIndex.from_frame(conflits))].drop_duplicates().sort_values(["PTID", "date_dx"])
dxb["code_dx"] = dxb["code_dx"].astype(int)
dxb["rang"] = dxb.groupby("PTID").cumcount()
if dxb.empty:
    raise ValueError("Aucune visite diagnostique retenue : verifie le format des PTID "
                     "et des dates dans DXSUM.")

nb_visites = dxb.groupby("PTID")["rang"].max() + 1
eligibles = nb_visites[nb_visites >= K + 1].index
d = dxb.loc[dxb["PTID"].isin(eligibles)]

ancre = (d.loc[d["rang"] == K - 1]
         .set_index("PTID")[["date_dx", "code_dx"]]
         .rename(columns={"date_dx": "date_Tk", "code_dx": "dx_Tk"}))

suite = d.merge(ancre, on="PTID")
suite["delai"] = (suite["date_dx"] - suite["date_Tk"]).dt.days
cand = suite.loc[suite["delai"].between(*FENETRE)].copy()
cand["distance"] = (cand["delai"] - HORIZON).abs()
cible_l = (cand.sort_values(["PTID", "distance", "date_dx"])
           .drop_duplicates("PTID")[["PTID", "code_dx", "delai"]]
           .rename(columns={"code_dx": "dx_cible_l", "delai": "delai_l"}))

base_l = ancre.reset_index().merge(cible_l, on="PTID", how="inner")
base_l = base_l.loc[base_l["dx_Tk"].isin([1, 2])].copy()     # pas deja dement a Tk
base_l["cible"] = (base_l["dx_cible_l"] > base_l["dx_Tk"]).astype(int)

print(f"Participants avec au moins {K + 1} visites : {len(eligibles)}")
print(f"... dont une visite cible dans la fenetre  : {len(ancre.reset_index().merge(cible_l, on='PTID'))}")
print(f"... et pas deja dements a Tk               : {len(base_l)}")
if len(base_l) < 100:
    raise ValueError(
        f"Seulement {len(base_l)} participants longitudinaux : trop peu pour conclure. "
        f"Essaie K=2 au lieu de {K}, ou elargis la fenetre cible.")
print(f"Aggravations : {int(base_l['cible'].sum())} ({base_l['cible'].mean():.1%})")

In [ ]:
# --- attacher les scores cliniques a chacune des K visites ---
panel = (d.loc[d["rang"] < K, ["PTID", "date_dx", "code_dx", "rang"]]
         .loc[lambda t: t["PTID"].isin(set(base_l["PTID"]))]
         .sort_values(["PTID", "rang"]).copy())

def charger(prefixe, col_date, correspondances):
    f = sorted(TABLES.rglob(prefixe + "*.csv"))
    if not f:
        return None
    t = pd.read_csv(f[-1], dtype="string", encoding="utf-8-sig")
    t.columns = t.columns.str.strip().str.upper()
    t["PTID"] = t["PTID"].str.strip().str.upper()
    if col_date not in t.columns:
        return None
    t["date_mesure"] = pd.to_datetime(
        t[col_date].astype("string").str.strip().str.slice(0, 10),
        format="%Y-%m-%d", errors="coerce").dt.normalize()
    noms = []
    for brut, nom in correspondances.items():
        if brut in t.columns:
            t[nom] = pd.to_numeric(t[brut], errors="coerce")
            t[nom] = t[nom].where(t[nom] >= 0)   # -1 = valeur manquante dans ADNI
            noms.append(nom)
    if not noms:
        return None
    return (t.loc[t["PTID"].isin(set(panel["PTID"])) & t["date_mesure"].notna(),
                  ["PTID", "date_mesure"] + noms]
            .dropna(subset=noms, how="all")
            .drop_duplicates(subset=["PTID", "date_mesure"])
            .sort_values("date_mesure"))

for prefixe, cd, corr in [
        ("All_Subjects_MMSE_", "VISDATE", {"MMSCORE": "MMSE"}),
        ("All_Subjects_CDR_", "VISDATE", {"CDGLOBAL": "CDR_global", "CDRSB": "CDR_SB"}),
        ("All_Subjects_ADAS_", "VISDATE", {"TOTAL13": "ADAS13"}),
        ("All_Subjects_FAQ_", "VISDATE", {"FAQTOTAL": "FAQ"})]:
    t = charger(prefixe, cd, corr)
    if t is None:
        print("Source absente, ignoree :", prefixe)
        continue
    noms = [c for c in t.columns if c not in ("PTID", "date_mesure")]
    lien = pd.merge_asof(panel.sort_values("date_dx"), t,
                         by="PTID", left_on="date_dx", right_on="date_mesure",
                         direction="backward", tolerance=pd.Timedelta(days=90))
    panel = panel.merge(lien[["PTID", "rang"] + noms], on=["PTID", "rang"], how="left")

VAR_VISITE = ["code_dx"] + [c for c in SCORES if c in panel.columns]
print("Variables par visite :", VAR_VISITE)

ordre = sorted(base_l["PTID"])
cube = (panel.set_index(["PTID", "rang"])[VAR_VISITE]
        .reindex(pd.MultiIndex.from_product([ordre, range(K)], names=["PTID", "rang"]))
        .to_numpy(dtype=float)
        .reshape(len(ordre), K, len(VAR_VISITE)))
y_l = base_l.set_index("PTID").loc[ordre, "cible"].to_numpy(dtype=int)
print("Forme des sequences :", cube.shape)

In [ ]:
# --- variables statiques, mesurees au plus tard a Tk ---
stat = base_l.set_index("PTID").loc[ordre, ["date_Tk"]].copy()
cols_stat = []
for c in ["sexe", "education", "APOE_e4", "naissance"]:
    if c in cohorte.columns:
        stat = stat.join(cohorte.set_index("PTID")[c], how="left")
        cols_stat.append(c)
if "naissance" in cols_stat:
    stat["age_Tk"] = stat["date_Tk"].dt.year - stat["naissance"]
    stat.loc[~stat["age_Tk"].between(40, 120), "age_Tk"] = np.nan
    cols_stat = [c for c in cols_stat if c != "naissance"] + ["age_Tk"]
S = stat[cols_stat].to_numpy(dtype=float) if cols_stat else np.empty((len(ordre), 0))
communs = len(set(ordre) & set(cohorte["PTID"]))
print(f"Participants longitudinaux aussi presents dans la cohorte tabulaire : "
      f"{communs} / {len(ordre)}")
if S.size:
    print("Variables statiques :", cols_stat,
          f"| couverture : {100 * (1 - np.isnan(S).mean()):.0f} %")
    print("Les deux modeles compares ci-dessous recoivent exactement les memes variables")
    print("statiques : la mesure de l'apport du temps reste donc valide.")

X_statique = np.hstack([cube[:, -1, :], S])                      # la 3e visite seule
pentes = cube[:, -1, :] - cube[:, 0, :]                          # evolution entre V1 et V3
X_longitudinal = np.hstack([cube.reshape(len(ordre), -1), pentes, S])
print("Statique     :", X_statique.shape)
print("Longitudinal :", X_longitudinal.shape)

resultats_D = []
for libelle, Xm in [("D0 — une seule visite (statique)", X_statique),
                    ("D — 3 visites + pentes (longitudinal)", X_longitudinal)]:
    auc_D, oof_D = evaluer(Xm, y_l, noms=("Regression logistique", "Gradient boosting"))
    rD = resume(auc_D)
    best_D = max(rD.items(), key=lambda kv: kv[1][0])
    print(f"\n=== {libelle} ===")
    for n, (m, s) in rD.items():
        print(f"  {n:24s} {m:.3f} +/- {s:.3f}")
    resultats_D.append({"variante": libelle, "modele": best_D[0],
                        "roc_auc": round(best_D[1][0], 3),
                        "ecart_type": round(best_D[1][1], 3),
                        "participants": len(y_l),
                        "auc_par_pli": [round(v, 3) for v in auc_D[best_D[0]]]})

gain_D = resultats_D[1]["roc_auc"] - resultats_D[0]["roc_auc"]
plis_gagnes = sum(a > b for a, b in zip(resultats_D[1]["auc_par_pli"],
                                        resultats_D[0]["auc_par_pli"]))
print(f"\n>>> Apport du suivi dans le temps : {gain_D:+.3f} "
      f"(gagne {plis_gagnes}/5 plis)")
print("    Rappel question 2 (ancienne cohorte, 352 participants) : +0,053, 5/5 plis")
display(pd.DataFrame(resultats_D)[["variante", "modele", "roc_auc", "ecart_type",
                                   "participants"]])

## 7. Synthèse : qu'est-ce qui a vraiment servi ?

Le tableau rassemble les variantes **comparables entre elles** (même cohorte, même cible,
mêmes plis). Le volet temporel est à part, avec sa propre comparaison interne.

Règle de lecture, la même que pour le choix du modèle en soutenance : un écart plus petit
que l'écart-type entre les plis n'est **pas** une amélioration démontrée. On ne retient un
changement que s'il apporte un gain visible, ou s'il simplifie le modèle à note égale.

In [ ]:
tableau = pd.DataFrame(comparaison)
display(tableau)

seuil_bruit = tableau.loc[0, "ecart_type"]
print(f"Ecart-type entre les plis de la reference : {seuil_bruit:.3f}")
print("Un gain inferieur a cette valeur n'est pas une amelioration demontree.\n")

retenus = tableau.loc[tableau["ecart_vs_reference"] > 0]
if retenus.empty:
    print("-> Aucune variante tabulaire ne depasse la reference.")
    print("   Conclusion : sur ces donnees, le plafond ne vient pas du choix du modele.")
else:
    meilleure = retenus.sort_values("roc_auc", ascending=False).iloc[0]
    sur = "clairement" if meilleure["ecart_vs_reference"] > seuil_bruit else "faiblement"
    print(f"-> Meilleure variante tabulaire : {meilleure['variante']} "
          f"({meilleure['roc_auc']:.3f}, {meilleure['ecart_vs_reference']:+.3f}) — {sur} au-dessus.")

print(f"\n-> Suivi dans le temps : {gain_D:+.3f} sur {len(y_l)} participants "
      f"({plis_gagnes}/5 plis)")

fig, ax = plt.subplots(figsize=(8.5, 4.2), constrained_layout=True)
pos = np.arange(len(tableau))[::-1]
ax.barh(pos, tableau["roc_auc"], xerr=tableau["ecart_type"], capsize=4,
        color=["#4C72B0" if v >= 0 else "#B0B0B0" for v in tableau["ecart_vs_reference"]])
ax.set_yticks(pos); ax.set_yticklabels(tableau["variante"], fontsize=9)
ax.axvline(tableau.loc[0, "roc_auc"], color="#C44E52", ls=":", lw=1.5,
           label="reference notebook 19")
ax.set_xlim(0.5, 1.0); ax.set_xlabel("ROC-AUC (5 plis)")
ax.set_title("Apport de chaque changement, a cohorte et plis identiques")
for p, m, e in zip(pos, tableau["roc_auc"], tableau["ecart_type"]):
    ax.text(m + e + 0.008, p, f"{m:.3f}", va="center", fontsize=9)
ax.legend(loc="lower right")
plt.show()

## 8. Export

In [ ]:
SORTIE = MANIFESTES / ("ameliorations_20_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
SORTIE.mkdir(parents=True, exist_ok=False)

def sauver(df, nom):
    df.to_csv(SORTIE / nom, index=False, encoding="utf-8-sig")

sauver(tableau, "comparaison_variantes.csv")
sauver(pd.DataFrame(resultats_D).drop(columns=["auc_par_pli"]), "volet_longitudinal.csv")
if not sous_groupes.empty:
    sauver(sous_groupes, "modeles_par_groupe_CN_MCI.csv")

bilan = {
    "experience": "20_ameliorations",
    "date": datetime.now().isoformat(),
    "export_18": EXPORT18.name,
    "reference_notebook_19": {"modele": REFERENCE[0], "roc_auc": round(REFERENCE[1][0], 3)},
    "variantes": tableau.to_dict(orient="records"),
    "longitudinal": {"participants": int(len(y_l)),
                     "positifs": int(y_l.sum()),
                     "statique": resultats_D[0]["roc_auc"],
                     "longitudinal": resultats_D[1]["roc_auc"],
                     "gain": round(float(gain_D), 3),
                     "plis_gagnes_sur_5": int(plis_gagnes)},
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
}
(SORTIE / "bilan.json").write_text(json.dumps(bilan, ensure_ascii=False, indent=2),
                                   encoding="utf-8")

print("=" * 78)
print(f">>> Reference notebook 19 : {REFERENCE[1][0]:.3f}")
print(f">>> Meilleure variante tabulaire : {tableau.sort_values('roc_auc').iloc[-1]['variante']} "
      f"= {tableau['roc_auc'].max():.3f}")
print(f">>> Suivi dans le temps : {resultats_D[0]['roc_auc']:.3f} -> "
      f"{resultats_D[1]['roc_auc']:.3f} ({gain_D:+.3f}) sur {len(y_l)} participants")
print("=" * 78)
print("\nResultats enregistres dans :", SORTIE)